# IMPORT LIBRARIES & LOAD DATASET

In [2]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [4]:
womens_world_cup_stats = pd.read_csv(
    '/Users/gui/womens_world_cup_2023_statsbomb_plusplus.csv'
)

# FILTER ROUND OF 16

In [9]:
r16 = womens_world_cup_stats[
    womens_world_cup_stats["stage"] == "Round of 16"
].copy()

r16.shape

(16, 53)

In [23]:
r16.columns.tolist()

['match_id',
 'match_date',
 'team',
 'opponent',
 'stage',
 'goals_for',
 'goals_against',
 'result',
 'points',
 'possession_pct',
 'passes',
 'pass_accuracy_pct',
 'shots',
 'shots_on_target',
 'shot_accuracy_pct',
 'xG',
 'xG_per_shot',
 'goals_minus_xG',
 'conversion_pct',
 'big_chances',
 'key_passes',
 'corners',
 'crosses',
 'progressive_passes',
 'progressive_carries',
 'final_third_entries',
 'box_entries',
 'high_turnovers',
 'shots_after_turnover',
 'goals_after_turnover',
 'set_piece_goals',
 'tackles',
 'tackles_won',
 'interceptions',
 'blocks',
 'clearances',
 'recoveries',
 'saves',
 'clean_sheet',
 'formation',
 'shot_assists',
 'through_balls',
 'cutbacks',
 'dribbles',
 'successful_dribbles',
 'dribble_success_pct',
 'duels',
 'duels_won',
 'counterpress_actions',
 'goals_from_shots',
 'own_goals_for',
 'own_goals_against',
 'goal_type',
 'efficiency',
 'efficiency_pct',
 'efficiency_score']

## Normalize metrics to 0–100

In [84]:
def min_max_score(series):
    if series.max() == series.min():
        return 50
    return ((series - series.min()) / (series.max() - series.min())) * 100

# CREATE THE SCORE OF THE 5 METRICS

In [86]:
r16["xG_score"] = min_max_score(
    r16["xG"]
)

r16["pass_accuracy_score"] = min_max_score(
    r16["pass_accuracy_pct"]
)

r16["possession_score"] = min_max_score(
    r16["possession_pct"]
)

r16["goals_scored_score"] = min_max_score(
    r16["goals_for"]
)

r16["goals_conceded_score"] = (
    100 - min_max_score(r16["goals_against"])
)

# PERFORMANCE SCORE

In [88]:
r16["performance_score"] = (
    r16["xG_score"] * 0.25 +
    r16["pass_accuracy_score"] * 0.25 +
    r16["possession_score"] * 0.10 +
    r16["goals_scored_score"] * 0.25 +
    r16["goals_conceded_score"] * 0.15
)

In [90]:
r16["performance_score"] = r16["performance_score"].round(1)

# FINAL TABLE

In [92]:
performance_table = r16[
    [
        "team",
        "xG",
        "pass_accuracy_pct",
        "possession_pct",
        "goals_for",
        "goals_against",
        "performance_score"
    ]
].copy()



performance_table = performance_table.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



performance_table.insert(
    0,
    "Rank",
    range(1, len(performance_table) + 1)
)



performance_table["xG"] = (
    performance_table["xG"].round(1)
)

performance_table["pass_accuracy_pct"] = (
    performance_table["pass_accuracy_pct"].round(1)
)

performance_table["possession_pct"] = (
    performance_table["possession_pct"].round(1)
)



performance_table = performance_table.rename(columns={
    "team": "Team",
    "xG": "xG (%)",
    "pass_accuracy_pct": "Pass Accuracy (%)",
    "possession_pct": "Possession (%)",
    "goals_for": "Goals Scored",
    "goals_against": "Goals Conceded",
    "performance_score": "Performance Score"
})



performance_table

,Rank,Team,xG (%),Pass Accuracy (%),Possession (%),Goals Scored,Goals Conceded,Performance Score
0,1,Spain Women's,2.8,85.8,68.9,5,1,81.4
1,2,France Women's,1.8,81.9,73.4,4,0,73.9
2,3,United States Women's,6.8,75.0,60.3,0,0,64.4
3,4,Netherlands Women's,1.4,82.0,70.8,2,0,61.8
4,5,Japan Women's,1.0,84.7,59.2,3,1,61.7
5,6,England Women's,4.7,78.4,49.9,0,0,56.7
6,7,Sweden Women's,6.0,71.4,39.7,0,0,54.4
7,8,Nigeria Women's,4.2,69.1,50.1,0,0,48.5
8,9,Australia Women's,0.8,69.5,46.0,2,0,45.4
9,10,Colombia Women's,1.1,67.2,47.0,1,0,40.1


## Table 2: Score Breakdown

In [94]:
# ==========================================
# SCORE BREAKDOWN
# ==========================================

score_breakdown = r16[
    [
        "team",
        "xG_score",
        "pass_accuracy_score",
        "possession_score",
        "goals_scored_score",
        "goals_conceded_score",
        "performance_score"
    ]
].copy()



score_breakdown = score_breakdown.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



score_breakdown.insert(
    0,
    "Rank",
    range(1, len(score_breakdown) + 1)
)



score_columns = [
    "xG_score",
    "pass_accuracy_score",
    "possession_score",
    "goals_scored_score",
    "goals_conceded_score",
    "performance_score"
]

score_breakdown[score_columns] = (
    score_breakdown[score_columns].round(1)
)



score_breakdown = score_breakdown.rename(columns={
    "team": "Team",
    "xG": "xG (25%)",
    "pass_accuracy_score": "Pass Accuracy Score (25%)",
    "possession_score": "Possession Score (10%)",
    "goals_scored_score": "Goals Scored Score (25%)",
    "goals_conceded_score": "Goals Conceded Score (15%)",
    "performance_score": "Performance Score"
})



score_breakdown

,Rank,Team,xG_score,Pass Accuracy Score (25%),Possession Score (10%),Goals Scored Score (25%),Goals Conceded Score (15%),Performance Score
0,1,Spain Women's,41.3,100.0,90.5,100.0,80.0,81.4
1,2,France Women's,26.7,88.8,100.0,80.0,100.0,73.9
2,3,United States Women's,100.0,68.9,72.0,0.0,100.0,64.4
3,4,Netherlands Women's,20.2,89.2,94.4,40.0,100.0,61.8
4,5,Japan Women's,14.1,96.8,69.7,60.0,80.0,61.7
5,6,England Women's,68.4,78.6,49.9,0.0,100.0,56.7
6,7,Sweden Women's,87.8,58.7,28.0,0.0,100.0,54.4
7,8,Nigeria Women's,61.9,52.0,50.1,0.0,100.0,48.5
8,9,Australia Women's,11.9,53.1,41.4,40.0,100.0,45.4
9,10,Colombia Women's,16.5,46.6,43.7,20.0,100.0,40.1


# Performance Score — Key Findings

The Performance Score provides a composite view of team performance in the Round of 16, combining attacking output, passing accuracy, possession and defensive results. Spain and France recorded the highest scores, reflecting their strong balance across the different dimensions of performance.

Importantly, the ranking also shows that possession alone does not determine performance. Teams such as Japan achieved a high score despite having less possession than some of their opponents, highlighting the importance of efficiency, execution and goal output.

The score should therefore be interpreted as a measure of **overall match performance**, rather than simply the match result. This allows differences in playing style to be captured while maintaining a strong focus on effectiveness and outcomes.